# Experiment 1 · Notebook 02: Cleaning and label preparation

*J26-IT-304 PP1. Baseline confusion classifier (TF-IDF + Logistic Regression). Sections 4–5.*

Applies the decisions from 01 §3.8 as ruled: deduplication, the group key, and the binary label mapping with its proxy caveat. No split, no model.

Section numbers run continuously across the Experiment 1 series, so a reference such as "EDA §3.5" or "section 8" points to the same place in every notebook:

| Notebook | Sections | Stage |
|---|---|---|
| `01_data_and_eda` | §1–§3 | provenance, load, exploratory analysis, leakage demonstration |
| `02_cleaning_and_labels` | §4–§5 | deduplication, group key, label mapping, proxy caveat |
| `03_split_and_baselines` | §6–§7 | group-aware split, trivial baselines, frozen OOD probe |
| `04_model_training_and_evaluation` | §8–§9 | TF-IDF + LR, group-aware CV, the single test evaluation |

Each notebook runs on its own from a clean kernel (kernel `exp1-confusion`). Its first code cell rebuilds its inputs with the same `confusion_baseline` functions the previous stage used, so the stages cannot drift apart. Run them in order 01 → 04 to regenerate every file in `reports/experiments/`.

**Setup.** Imports, and the input of this stage: the raw dataset at the pinned revision, with `FollowUp` removed exactly as in notebook 01.

In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from confusion_baseline import SEED
from confusion_baseline.data import DATASET_REVISION, load_raw

pd.set_option("display.width", 200)
CLASSES = ["Correct", "Partial Correct", "Wrong"]

# Input of this stage: the raw dataset at the pinned revision (notebook 01, section 2).
df, followup_quarantine, load_report = load_raw()
del followup_quarantine          # FollowUp is only ever read by the leakage demonstration in 01 §3.7
assert not [c for c in df.columns if "follow" in c.lower()]
print("dataset revision:", DATASET_REVISION)
print("input frame     :", df.shape, list(df.columns))
print("PASS: no FollowUp-derived column in the working frame")

dataset revision: e29cfdd4ec4179f54b2ea0409329207104de8bed
input frame     : (1078, 4) ['row_id', 'Question', 'Answer', 'Correctness']
PASS: no FollowUp-derived column in the working frame


## 4. Cleaning

**Why.** Each step follows from the EDA (section 3.8), as ruled by the supervisor. Dedup runs
**before** any split, so copies of one record cannot land on both sides. The model consumes the **raw
`Answer` text**: no custom preprocessor goes into the sklearn Pipeline, so it unpickles with sklearn
alone and the tutor service passes raw text. `clean()` lives in `confusion_baseline/prep.py` so notebook
and training script share it. `Question` is kept because it is the group key and is needed for error
analysis, but it is **never a model input**: the question is the tutor's preceding turn, which is
conversational context, and context is the PP2 research question.

In [2]:
from confusion_baseline.prep import clean, map_labels, save_processed, LABEL_MAP_PRIMARY, LABEL_MAP_DROP_PARTIAL, POS_LABEL

clean_df, attrition = clean(df)

table = pd.DataFrame(attrition, columns=["step", "rows after", "note"])
table.insert(2, "rows removed", (table["rows after"].shift(1, fill_value=table["rows after"].iloc[0]) - table["rows after"]).astype(int))
pd.set_option("display.max_colwidth", 100)
print("ATTRITION TABLE")
display(table[["step", "rows after", "rows removed", "note"]])

ATTRITION TABLE


,step,rows after,rows removed,note
0,loaded (FollowUp already removed),1078,0,"columns: ['row_id', 'Question', 'Answer', 'Correctness']"
1,missing-value check,1078,0,"0 rows with a missing ['Question', 'Answer', 'Correctness'] value dropped"
2,"dedup on (Question, Answer, Correctness)",1076,2,"2 duplicate rows dropped, lowest row_id kept; label conflicts = 0"


**Why.** The attrition table says how many rows went; this shows *which*. The removed `row_id`s must be
exactly the later copies of the record found in 3.3, and the label-conflict count must be 0 (otherwise
dedup on a key that includes `Correctness` would hide a contradiction).

In [3]:
removed = sorted(set(df["row_id"]) - set(clean_df["row_id"]))
print("removed row_ids:", removed)
print("kept copy of that record, row_id:", sorted(set(clean_df["row_id"]) & {0, 15, 1060}))
nl = clean_df.groupby(["Question", "Answer"])["Correctness"].nunique()
n_conf = int((nl > 1).sum())
print("label conflicts (same Question+Answer, different Correctness):", n_conf)
assert n_conf == 0
assert removed == [15, 1060], removed
print("PASS: removed exactly the surplus copies of the record at row_ids 0 / 15 / 1060; 0 label conflicts")

removed row_ids: [15, 1060]
kept copy of that record, row_id: [0]
label conflicts (same Question+Answer, different Correctness): 0
PASS: removed exactly the surplus copies of the record at row_ids 0 / 15 / 1060; 0 label conflicts


**Why. No text normalisation in the model path.** Section 3.5 proposed normalising curly quotes,
backticks and whitespace. The supervisor's verification found this changes no TF-IDF token. This cell
reproduces that proof: for four analyzer configurations (`TfidfVectorizer(...).build_analyzer()` builds
the tokenizer only and **fits nothing**), it counts answers whose token list differs between the raw text
and a normalised copy (curly quotes -> ASCII, backticks stripped, whitespace collapsed). The
normaliser here exists only for this check and is **not** part of the package or the model. The expected
count is 0 in every row; anything else means the ruling does not hold and must be reported.

In [4]:
def _probe_normalise(t):
    for a, b in (("\u2018", "'"), ("\u2019", "'"), ("\u201c", '"'), ("\u201d", '"')):
        t = t.replace(a, b)
    return re.sub(r"\s+", " ", t.replace("`", "")).strip()

raw = clean_df["Answer"]
norm = raw.map(_probe_normalise)
rows = []
for ngram in [(1, 1), (1, 2)]:
    for stop in [None, "english"]:
        analyzer = TfidfVectorizer(ngram_range=ngram, stop_words=stop).build_analyzer()
        n_diff = int(sum(analyzer(a) != analyzer(b) for a, b in zip(raw, norm)))
        rows.append({"ngram_range": ngram, "stop_words": stop, "answers with different tokens": n_diff, "of": len(raw)})
check = pd.DataFrame(rows)
display(check)
assert (check["answers with different tokens"] == 0).all(), "normalisation changes tokens: stop and report"
print("PASS: normalisation changes the token list of 0 answers in all 4 analyzer configs")
print("answers where raw text != normalised text (string level):", int((raw != norm).sum()))

,ngram_range,stop_words,answers with different tokens,of
0,"(1, 1)",NaN,0,1076
1,"(1, 1)",english,0,1076
2,"(1, 2)",NaN,0,1076
3,"(1, 2)",english,0,1076


PASS: normalisation changes the token list of 0 answers in all 4 analyzer configs
answers where raw text != normalised text (string level): 564


**Why.** The one artefact the EDA flagged is non-ASCII text. This cell lists every non-ASCII character
in `Answer` and how common it is per class, so the finding is exact and not inferred.

In [5]:
chars = Counter()
for t in clean_df["Answer"]:
    chars.update(c for c in t if ord(c) > 127)
print("non-ASCII characters in Answer:", {f"U+{ord(c):04X} {c!r}": n for c, n in chars.items()})
has = clean_df["Answer"].str.contains("\u2019", regex=False)
print("answers containing U+2019, per class (proportion):")
print(has.groupby(clean_df["Correctness"]).mean().reindex(CLASSES).round(3).to_string())
print("non-ASCII characters in Question:", sum(not q.isascii() for q in clean_df["Question"]), "questions")

non-ASCII characters in Answer: {"U+2019 '’'": 73}
answers containing U+2019, per class (proportion):
Correctness
Correct            0.036
Partial Correct    0.073
Wrong              0.082
non-ASCII characters in Question: 6 questions


**Finding.** The only non-ASCII character in the answers is **U+2019 (the curly apostrophe)**, present in
3.6% / 7.3% / 8.2% of Correct / Partial / Wrong answers. It is a generator artefact. Word-level TF-IDF with
sklearn's default token pattern cannot see it (the check above shows 0 token differences), so it cannot leak
the label through word features. **Character n-grams would see it**, and could pick up this artefact as
a class signal. For that reason **character n-grams are excluded from all later configurations**.

**Why.** The group key is what the next unit splits on, so a question's answers stay on one side. It is
`normalize_key(Question)`. The split itself will be `GroupShuffleSplit` per the spec, **not**
stratified.

In [6]:
print("group_key unique count:", clean_df["group_key"].nunique(), "(expected 360)")
print("rows per group_key (min / median / max):", clean_df.groupby("group_key").size().agg(["min", "median", "max"]).tolist())
assert clean_df["group_key"].nunique() == 360
print("columns:", list(clean_df.columns))

group_key unique count: 360 (expected 360)
rows per group_key (min / median / max): [1.0, 3.0, 12.0]
columns: ['row_id', 'Question', 'Answer', 'Correctness', 'group_key']


## 5. Label preparation

**The mapping is the baseline operationalisation, not a definition of confusion.**
Primary mapping: `Correct` -> **understood**; `Partial Correct` and `Wrong` -> **confused** (positive class =
confused). A second mapping, `drop_partial`, removes `Partial Correct` rows and keeps `Correct` ->
understood, `Wrong` -> confused. Both are defined once in `confusion_baseline/prep.py` and will be written
into the model's metadata.

**Proxy caveat.** The label is correctness, used as a stand-in for confusion:

| Correctness tells you | Confusion is |
|---|---|
| Whether the answer matched the expected content | Whether the learner knows they do not understand |
| Observable from the answer alone | A cognitive state the answer only hints at |
| What this dataset labels | What the deployed system must detect |

A confidently wrong answer is not confusion; a student who says "wait, I don't follow" is confused but has
produced no answer to mark. The deployed system will mostly see the second kind, and this dataset contains
none (EDA section 3.5: 0 hedging words in 1,078 answers). This is why the system pairs the model with a
rule layer, and why the OOD probe exists.

**Why.** Both mappings are applied and compared here. For each, the share of the positive class sets the
floor the model must beat: a constant "always confused" predictor scores F1(confused) = 2p/(1+p).

In [7]:
y_primary = map_labels(clean_df, "primary")
y_drop = map_labels(clean_df, "drop_partial")
print("LABEL_MAP_PRIMARY       :", LABEL_MAP_PRIMARY)
print("LABEL_MAP_DROP_PARTIAL  :", LABEL_MAP_DROP_PARTIAL)
print("POS_LABEL               :", POS_LABEL)
print()
for name, d in [("primary", y_primary), ("drop_partial", y_drop)]:
    vc = d["label"].value_counts()
    p = float(d["y"].mean())
    print(f"--- mapping: {name}   rows = {len(d)}   (Partial Correct rows dropped: {d.attrs['dropped_partial']})")
    display(pd.DataFrame({"count": vc, "proportion": (vc / len(d)).round(4)}))
    print(f"    p(confused) = {p:.4f}   constant 'always confused' F1(confused) = 2p/(1+p) = {2 * p / (1 + p):.4f}")

LABEL_MAP_PRIMARY       : {'Correct': 'understood', 'Partial Correct': 'confused', 'Wrong': 'confused'}
LABEL_MAP_DROP_PARTIAL  : {'Correct': 'understood', 'Wrong': 'confused'}
POS_LABEL               : confused

--- mapping: primary   rows = 1076   (Partial Correct rows dropped: 0)


,count,proportion
label,,
confused,656,0.6097
understood,420,0.3903


    p(confused) = 0.6097   constant 'always confused' F1(confused) = 2p/(1+p) = 0.7575
--- mapping: drop_partial   rows = 787   (Partial Correct rows dropped: 289)


,count,proportion
label,,
understood,420,0.5337
confused,367,0.4663


    p(confused) = 0.4663   constant 'always confused' F1(confused) = 2p/(1+p) = 0.6360


**Why.** The primary-mapped frame is the single input for the split in the next unit. It is written to
`data/processed/` (gitignored, since the dataset licence is unstated), and the assertions below are the
contract the next unit relies on.

In [8]:
path = save_processed(y_primary, "clean_primary")
print("saved:", path.relative_to(path.parents[2]).as_posix())

assert len(y_primary) == 1076, len(y_primary)
assert y_primary["group_key"].nunique() == 360
expected = ["row_id", "Question", "Answer", "Correctness", "group_key", "label", "y"]
assert list(y_primary.columns) == expected, list(y_primary.columns)
print("PASS: primary frame has", len(y_primary), "rows;", y_primary["group_key"].nunique(), "groups; columns", list(y_primary.columns))

saved: data/processed/clean_primary.parquet
PASS: primary frame has 1076 rows; 360 groups; columns ['row_id', 'Question', 'Answer', 'Correctness', 'group_key', 'label', 'y']


## Findings of notebook 02

| Topic | Finding | Where |
|---|---|---|
| Attrition | 1,078 → 1,076: 0 rows with missing values; dedup removed rows 15 and 1060 (copy 0 kept); 0 label conflicts. | §4 |
| Normalisation | It changes the TF-IDF tokens of **0 of 1,076** answers in all 4 analyzer settings. The model therefore takes the **raw answer text**, and the pickled pipeline needs only sklearn. | §4 |
| Artefact | U+2019 is the only non-ASCII character in answers (Correct 3.6% · Partial 7.3% · Wrong 8.2%). Word-level TF-IDF cannot see it, and character n-grams are excluded for that reason. | §4 |
| Groups | 360 question groups (`group_key = normalize_key(Question)`). The question is never a model input: it is the tutor's preceding turn, and conversational context is the PP2 research question. | §4 |
| Primary mapping | Correct → understood; Partial Correct + Wrong → confused: 656 confused / 420 understood (60.97%). Floor: "always confused" F1(confused) = **0.7575**. | §5 |
| Sensitivity mapping | Partial Correct dropped: 787 rows, 46.6% confused, floor 0.6360. Used by the sensitivity check. | §5 |
| Proxy caveat | The label is answer **correctness**, a proxy for confusion and not a measurement of it. A confidently wrong answer is not confusion, and "I don't follow" is confusion with no answer to mark. | §5 |